In [ ]:
from timm import create_model
from timm.models import ResNet
import torch

In [ ]:
import main_helper

In [ ]:
import importlib
importlib.reload(main_helper)

In [ ]:
import argparse

config = argparse.Namespace(
    dataset='CIFAR10',
    model='VIT_Tiny',
    batch_size=128,
    learning_rate=0.001,
    epoch=50,
    technique='LoRITa',
    rectangular_mode=True,
    target_rank=4,
    lmbda=0.01,
    depth_lorita=4,
    weight_decay_alpha=0.00,
    dropout=0.0,
    threads=4,
    augmentation=False,
    target_modules=['Q', 'K', 'V','FC1','FC2'],
    DEVICE = torch.device("cuda") 
)

In [ ]:
model = main_helper.create_model('timm/vit_tiny_patch16_224', pretrained=False, num_classes=10)
print(model)

In [ ]:
model = main_helper.model_parser((-1,3,224,224), 10, config)
regularizer =  main_helper.configure_model_experiment(model,config)
main_helper.get_model_memory(model)

In [ ]:
cloned_config = argparse.Namespace(
    dataset='CIFAR10',
    model='VIT_Tiny',
    batch_size=128,
    learning_rate=0.001,
    epoch=50,
    technique='Truncate',
    rectangular_mode=True,
    target_rank=4,
    lmbda=0.01,
    depth_lorita=2,
    weight_decay_alpha=0.00,
    dropout=0.0,
    threads=4,
    augmentation=False,
    target_modules=['Q', 'K', 'V','FC1','FC2'],
    DEVICE = torch.device("cuda") 
)

In [ ]:
print(model)

In [ ]:
random_vector = torch.randn(192)
random_vector = random_vector.to(next(model.parameters()).device)

In [ ]:
lorita_result = model.blocks[0].attn.qkv(random_vector)
fc1_lorita_result = model.blocks[0].mlp.fc1(random_vector)

In [ ]:
regularizer =  main_helper.configure_model_experiment(model,cloned_config)
main_helper.get_model_memory(model)

In [ ]:
simplify_result = model.blocks[0].attn.qkv(random_vector)
fc1_simplify_result = model.blocks[0].mlp.fc1(random_vector)

In [ ]:
lorita_result-simplify_result

In [ ]:
fc1_lorita_result-fc1_simplify_result

In [ ]:
model()